In [1]:

%pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0" "pandas==2.2.3"


In [2]:

import google.genai
import google.auth
import pandas as pd

print("Google GenAI version:", google.genai.__version__)
print("Google Auth version:", google.auth.__version__)
print("Pandas version:", pd.__version__)

print("Environment setup completed!")


Google GenAI version: 2.12.1
Google Auth version: 2.49.0
Pandas version: 2.2.3
Environment setup completed!


In [34]:
!pip install -q "google-genai>=2.12.1,<3"

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL_ID,
    contents="You are an AI Lead Qualification Agent. Explain how you qualify a sales lead in three simple sentences."
)

print(response.text)

First, I analyze the prospect's profile against your ideal customer criteria to determine if they have a genuine need for your solution. Next, I verify their authority and budget to ensure they have both the influence and the resources to make a purchase. Finally, I identify their current timeline and pain points to prioritize them as a high-value opportunity for your sales team.


In [5]:
import pandas as pd

lead = {
    "name": input("Enter lead name: "),
    "email": input("Enter email: "),
    "company": input("Enter company name: "),
    "industry": input("Enter industry: "),
    "requirement": input("Enter requirement: "),
    "budget": input("Enter budget: "),
    "timeline": input("Enter expected timeline: ")
}

lead_df = pd.DataFrame([lead])

print("\nLead Details:")
display(lead_df)

Enter lead name: sivanaga
Enter email: naga@gmail.com
Enter company name: splenden
Enter industry: it
Enter requirement: Wants to develop an AI-powered chatbot to automate customer support queries on the company website.
Enter budget: ₹1,50,000
Enter expected timeline: Within 30 days

Lead Details:


,name,email,company,industry,requirement,budget,timeline
0,sivanaga,naga@gmail.com,splenden,it,Wants to develop an AI-powered chatbot to auto...,"₹1,50,000",Within 30 days


In [6]:
# AI Lead Qualification Prompt

prompt = f"""
You are an expert B2B AI Lead Qualification Agent.

Analyze only the current lead's information.

LEAD DETAILS:
Name: {lead['name']}
Email: {lead['email']}
Company: {lead['company']}
Industry: {lead['industry']}
Requirement: {lead['requirement']}
Budget: {lead['budget']}
Timeline: {lead['timeline']}

TASK:
Evaluate this lead based on:
1. Requirement clarity
2. Purchase intent
3. Budget
4. Implementation timeline
5. Business need

Provide the following report:

1. Lead Score: 0-100
2. Lead Category: Hot, Warm, or Cold
3. Purchase Intent
4. Budget Suitability
5. Requirement Analysis
6. Reason for Score
7. Recommended Next Action

IMPORTANT RULES:
- Use only the current lead details.
- Never use information from previous leads.
- Do not assume missing information.
- Do not reduce the score simply because the email uses Gmail or another personal email provider.
- Do not claim that a budget is sufficient without considering project scope.
- Return a clear and professional report.
"""

print("AI qualification prompt created successfully!")

AI qualification prompt created successfully!


In [7]:
available_models = []

for model in client.models.list():
    if "generateContent" in (model.supported_actions or []):
        available_models.append(model.name)

print("Available Gemini models:")
for model in available_models:
    print(model)

Available Gemini models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/ge

In [8]:

import time


# Generate a fresh AI qualification report

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

new_report = response.text

print("========== NEW AI LEAD QUALIFICATION REPORT ==========")
print(new_report)
print("======================================================")


========== NEW AI LEAD QUALIFICATION REPORT ==========
### Lead Qualification Report

**1. Lead Score:** 85/100

**2. Lead Category:** Hot

**3. Purchase Intent:** High. The lead has defined a specific use case (AI-powered customer support chatbot) and provided clear constraints (budget and timeline).

**4. Budget Suitability:** Pending verification. The budget of ₹1,50,000 is specific, but whether this is sufficient depends on the complexity of the AI model, integration requirements with existing systems, and maintenance scope.

**5. Requirement Analysis:** The requirement is clear and actionable. The lead is looking for an AI solution to automate customer support on their website, indicating a direct need to improve operational efficiency.

**6. Reason for Score:** 
*   **Strengths:** The lead provided a concrete business problem, a defined budget, and a strict 30-day implementation timeline. This indicates the lead is in the active evaluation or decision-making stage.
*   **Weakness

In [ ]:
print(lead)

In [9]:

import sqlite3
import re
import pandas as pd
from datetime import datetime

# Connect to SQLite database
conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

# Create leads table
cursor.execute("""
CREATE TABLE IF NOT EXISTS leads (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    email TEXT UNIQUE,
    company TEXT,
    industry TEXT,
    requirement TEXT,
    budget TEXT,
    timeline TEXT,
    lead_score INTEGER,
    lead_category TEXT,
    ai_report TEXT,
    created_at TEXT
)
""")

# Extract score and category from Gemini response
ai_report = response.text

score_match = re.search(r"Lead Score:\*?\*?\s*(\d+)", ai_report, re.I)
category_match = re.search(
    r"Lead Category:\*?\*?\s*(Hot|Warm|Cold)",
    ai_report,
    re.I
)

lead_score = int(score_match.group(1)) if score_match else None
lead_category = category_match.group(1).capitalize() if category_match else None

# Save lead and AI analysis
cursor.execute("""
INSERT INTO leads (
    name, email, company, industry, requirement,
    budget, timeline, lead_score, lead_category,
    ai_report, created_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(email) DO UPDATE SET
    name=excluded.name,
    company=excluded.company,
    industry=excluded.industry,
    requirement=excluded.requirement,
    budget=excluded.budget,
    timeline=excluded.timeline,
    lead_score=excluded.lead_score,
    lead_category=excluded.lead_category,
    ai_report=excluded.ai_report
""", (
    lead["name"],
    lead["email"],
    lead["company"],
    lead["industry"],
    lead["requirement"],
    lead["budget"],
    lead["timeline"],
    lead_score,
    lead_category,
    ai_report,
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
))

conn.commit()

# Display saved leads
saved_leads = pd.read_sql_query(
    "SELECT * FROM leads ORDER BY id DESC",
    conn
)

print("Lead saved successfully!")
display(saved_leads)


Lead saved successfully!


,id,name,email,company,industry,requirement,budget,timeline,lead_score,lead_category,ai_report,created_at
0,1,sivanaga,naga@gmail.com,splenden,it,Wants to develop an AI-powered chatbot to auto...,"₹1,50,000",Within 30 days,85,Hot,### Lead Qualification Report\n\n**1. Lead Sco...,2026-11-21 03:22:00


In [10]:
print("CURRENT PROMPT:")
print(prompt)

print("\nCURRENT LEAD:")
print(lead)

CURRENT PROMPT:

You are an expert B2B AI Lead Qualification Agent.

Analyze only the current lead's information.

LEAD DETAILS:
Name: sivanaga
Email: naga@gmail.com
Company: splenden
Industry: it
Requirement: Wants to develop an AI-powered chatbot to automate customer support queries on the company website.
Budget: ₹1,50,000
Timeline: Within 30 days

TASK:
Evaluate this lead based on:
1. Requirement clarity
2. Purchase intent
3. Budget
4. Implementation timeline
5. Business need

Provide the following report:

1. Lead Score: 0-100
2. Lead Category: Hot, Warm, or Cold
3. Purchase Intent
4. Budget Suitability
5. Requirement Analysis
6. Reason for Score
7. Recommended Next Action

IMPORTANT RULES:
- Use only the current lead details.
- Never use information from previous leads.
- Do not assume missing information.
- Do not reduce the score simply because the email uses Gmail or another personal email provider.
- Do not claim that a budget is sufficient without considering project scope.


In [11]:
!pip install -q streamlit

In [12]:

%%writefile app.py

import streamlit as st
import sqlite3
import pandas as pd

st.set_page_config(
    page_title="AI Lead Management Agent",
    page_icon="🎯",
    layout="wide"
)

st.title("🎯 Intelligent AI Lead Management Agent")
st.caption("AI-powered lead qualification and management dashboard")

# Connect to database
conn = sqlite3.connect("lead_management.db")

df = pd.read_sql_query(
    "SELECT * FROM leads ORDER BY id DESC",
    conn
)

conn.close()

if df.empty:
    st.warning("No leads available.")
else:
    # Dashboard metrics
    col1, col2, col3, col4 = st.columns(4)

    col1.metric("Total Leads", len(df))
    col2.metric("Hot Leads", len(df[df["lead_category"] == "Hot"]))
    col3.metric("Warm Leads", len(df[df["lead_category"] == "Warm"]))
    col4.metric("Cold Leads", len(df[df["lead_category"] == "Cold"]))

    st.divider()

    # Lead score chart
    st.subheader("Lead Score Overview")
    st.bar_chart(df.set_index("name")["lead_score"])

    # Filter leads
    st.subheader("Lead Management")

    category = st.selectbox(
        "Filter by Lead Category",
        ["All", "Hot", "Warm", "Cold"]
    )

    if category != "All":
        filtered_df = df[df["lead_category"] == category]
    else:
        filtered_df = df

    st.dataframe(
        filtered_df[
            ["name", "email", "company", "budget",
             "timeline", "lead_score", "lead_category"]
        ],
        use_container_width=True,
        hide_index=True
    )

    # Individual lead report
    st.subheader("AI Qualification Reports")

    selected_email = st.selectbox(
        "Select a lead",
        filtered_df["email"].tolist()
    )

    if selected_email:
        selected_lead = df[df["email"] == selected_email].iloc[0]

        st.write("**Lead:**", selected_lead["name"])
        st.write("**Company:**", selected_lead["company"])
        st.write("**Score:**", selected_lead["lead_score"])
        st.write("**Category:**", selected_lead["lead_category"])

        st.markdown(selected_lead["ai_report"])

st.success("Dashboard loaded successfully!")


Writing app.py


In [13]:
import os

print("app.py exists:", os.path.exists("app.py"))

if os.path.exists("app.py"):
    print("File size:", os.path.getsize("app.py"), "bytes")

app.py exists: True
File size: 2072 bytes


In [14]:
!pip install -q pyngrok

In [15]:
import subprocess
import time

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Streamlit process running:", process.poll() is None)

Streamlit process running: True


In [39]:
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get("NGROK_AUTH_TOKEN")
ngrok.set_auth_token(token)

print("ngrok authentication configured!")

PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'

In [40]:
from pyngrok import ngrok

# Create a tunnel to Streamlit
public_url = ngrok.connect(8501)

print("Your AI Lead Management Dashboard:")
print(public_url)

Your AI Lead Management Dashboard:
NgrokTunnel: "https://specks-doorman-turbulent.ngrok-free.dev" -> "http://localhost:8501"


In [17]:
from pyngrok import ngrok

ngrok.set_auth_token("3MMsYz5mNkfdVhGhm4ITJgALycf_6hmH8NzwXPt9ZRpTnffgo")

print("Ngrok authentication configured successfully!")

Ngrok authentication configured successfully!


In [18]:
from pyngrok import ngrok

# Remove any previous tunnel
ngrok.kill()

# Connect Streamlit port
public_url = ngrok.connect(8501, "http")

print("Your AI Lead Management Dashboard is ready!")
print(public_url)

Your AI Lead Management Dashboard is ready!
NgrokTunnel: "https://specks-doorman-turbulent.ngrok-free.dev" -> "http://localhost:8501"


In [19]:

# Generate a personalized follow-up email using Gemini

followup_prompt = f"""
You are a professional B2B sales assistant.

Create a personalized follow-up email for this lead.

Lead Name: {lead['name']}
Company: {lead['company']}
Industry: {lead['industry']}
Requirement: {lead['requirement']}
Budget: {lead['budget']}
Timeline: {lead['timeline']}

AI Qualification:
{response.text}

Instructions:
1. Write a professional and friendly email.
2. Include a suitable subject line.
3. Mention the customer's chatbot requirement.
4. Suggest scheduling a discovery call.
5. Keep the email concise and persuasive.
6. Do not invent company achievements, pricing promises,
   or meeting dates.
7. Do not make assumptions based on the email domain.

Return the subject and complete email body.
"""

followup_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=followup_prompt
)

followup_email = followup_response.text

print("========== AI GENERATED FOLLOW-UP EMAIL ==========")
print(followup_email)
print("==================================================")


========== AI GENERATED FOLLOW-UP EMAIL ==========
Subject: Moving forward: AI-powered chatbot solution for Splenden

Hi Sivanaga,

Thank you for reaching out regarding your interest in developing an AI-powered chatbot for Splenden. It is clear that automating your customer support queries is a priority, and I would love to help you bring this solution to life.

Given your target timeline of 30 days, I would like to schedule a brief discovery call to better understand your current website infrastructure and the specific volume of queries you are looking to automate. This will help us ensure that we can deliver a high-quality, effective solution that aligns with your requirements and project goals.

Are you available for a 15-minute call sometime this week to discuss the scope and technical requirements in more detail? Please let me know a time that works best for you, and I will send over a calendar invite.

I look forward to hearing from you.

Best regards,

[Your Name]
[Your Title]
[

In [20]:

import sqlite3

conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

# Add follow-up email column if it doesn't exist
columns = [
    row[1] for row in cursor.execute(
        "PRAGMA table_info(leads)"
    ).fetchall()
]

if "follow_up_email" not in columns:
    cursor.execute(
        "ALTER TABLE leads ADD COLUMN follow_up_email TEXT"
    )

# Save generated email
cursor.execute("""
UPDATE leads
SET follow_up_email = ?
WHERE email = ?
""", (followup_email, lead["email"]))

conn.commit()

print("AI follow-up email saved successfully!")

conn.close()


AI follow-up email saved successfully!


In [21]:
%%writefile -a app.py

# AI Follow-up Email Section
st.divider()
st.subheader("✉️ AI Generated Follow-up Emails")

if "follow_up_email" in df.columns:
    for _, row in df.iterrows():
        if pd.notna(row["follow_up_email"]) and row["follow_up_email"]:
            with st.expander(f"Email for {row['name']} - {row['company']}"):
                st.text_area(
                    "Generated Follow-up Email",
                    value=row["follow_up_email"],
                    height=300,
                    key=f"email_{row['id']}"
                )
else:
    st.info("No follow-up emails saved yet.")

Appending to app.py


In [22]:
import subprocess
import sys
import time

if process.poll() is None:
    process.terminate()
    process.wait()

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit",
        "run", "app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true"
    ],
    stdout=open("/tmp/streamlit.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Updated dashboard running:", process.poll() is None)

Updated dashboard running: True


In [23]:
import requests
import time

time.sleep(5)

try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Streamlit status:", r.status_code)
    print("Response:", r.text)

except Exception as e:
    print("Streamlit connection failed:", e)

Streamlit status: 200
Response: ok


In [24]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
    print("Status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("Server error:", e)

Status: 200
Response: ok


In [25]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501, "http")

print("Updated Dashboard URL:")
print(public_url.public_url)

Updated Dashboard URL:
https://specks-doorman-turbulent.ngrok-free.dev


In [26]:
import sqlite3

conn = sqlite3.connect("lead_management.db")

cursor = conn.cursor()

cursor.execute("""
SELECT name, email, follow_up_email
FROM leads
""")

for row in cursor.fetchall():
    print("Name:", row[0])
    print("Email:", row[1])
    print("Follow-up Email:", row[2])

conn.close()

Name: sivanaga
Email: naga@gmail.com
Follow-up Email: Subject: Moving forward: AI-powered chatbot solution for Splenden

Hi Sivanaga,

Thank you for reaching out regarding your interest in developing an AI-powered chatbot for Splenden. It is clear that automating your customer support queries is a priority, and I would love to help you bring this solution to life.

Given your target timeline of 30 days, I would like to schedule a brief discovery call to better understand your current website infrastructure and the specific volume of queries you are looking to automate. This will help us ensure that we can deliver a high-quality, effective solution that aligns with your requirements and project goals.

Are you available for a 15-minute call sometime this week to discuss the scope and technical requirements in more detail? Please let me know a time that works best for you, and I will send over a calendar invite.

I look forward to hearing from you.

Best regards,

[Your Name]
[Your Title

In [27]:
import sqlite3

conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

cursor.execute("""
SELECT id, name, email, company, budget, timeline, lead_score, lead_category
FROM leads
ORDER BY id
""")

for row in cursor.fetchall():
    print(row)

conn.close()

(1, 'sivanaga', 'naga@gmail.com', 'splenden', '₹1,50,000', 'Within 30 days', 85, 'Hot')


In [28]:
import sqlite3

conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

cursor.execute("""
SELECT name, company, ai_report
FROM leads
ORDER BY id
""")

for name, company, report in cursor.fetchall():
    print("\nLead:", name)
    print("Company:", company)
    print("AI Report:", report)
    print("-" * 50)

conn.close()


Lead: sivanaga
Company: splenden
AI Report: ### Lead Qualification Report

**1. Lead Score:** 85/100

**2. Lead Category:** Hot

**3. Purchase Intent:** High. The lead has defined a specific use case (AI-powered customer support chatbot) and provided clear constraints (budget and timeline).

**4. Budget Suitability:** Pending verification. The budget of ₹1,50,000 is specific, but whether this is sufficient depends on the complexity of the AI model, integration requirements with existing systems, and maintenance scope.

**5. Requirement Analysis:** The requirement is clear and actionable. The lead is looking for an AI solution to automate customer support on their website, indicating a direct need to improve operational efficiency.

**6. Reason for Score:** 
*   **Strengths:** The lead provided a concrete business problem, a defined budget, and a strict 30-day implementation timeline. This indicates the lead is in the active evaluation or decision-making stage.
*   **Weaknesses:** The 

In [30]:
import sqlite3

conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

cursor.execute("""
UPDATE leads
SET ai_report = ?
WHERE email = ?
""", (new_report, "sai@gmail.com"))

conn.commit()

# Verify the update
cursor.execute("""
SELECT name, company, ai_report
FROM leads
WHERE email = 'sai@gmail.com'
""")

result = cursor.fetchone()

print("Name:", result[0])
print("Company:", result[1])
print("Updated Report:")
print(result[2])

conn.close()

TypeError: 'NoneType' object is not subscriptable

In [31]:
# Generate a personalized follow-up email for Sai

followup_prompt = f"""
You are a professional B2B sales assistant.

Create a personalized follow-up email for this lead.

Lead Name: {lead['name']}
Company: {lead['company']}
Industry: {lead['industry']}
Requirement: {lead['requirement']}
Budget: {lead['budget']}
Timeline: {lead['timeline']}

AI Qualification Report:
{new_report}

Instructions:
1. Write a professional and friendly email.
2. Include a suitable subject line.
3. Mention the AI-powered chatbot requirement.
4. Suggest scheduling a discovery call.
5. Keep the email concise and persuasive.
6. Do not invent company achievements, pricing promises,
   or meeting dates.
7. Do not assume the lead's authority or project scale.

Return the subject and complete email body.
"""

followup_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=followup_prompt
)

sai_followup_email = followup_response.text

print("========== SAI'S FOLLOW-UP EMAIL ==========")
print(sai_followup_email)
print("===========================================")

========== SAI'S FOLLOW-UP EMAIL ==========
Subject: Following up: AI-powered chatbot for Splenden

Hi Sivanaga,

Thank you for reaching out to discuss your interest in implementing an AI-powered chatbot for Splenden. 

Automating customer support queries is a great way to boost operational efficiency, and I would love to learn more about your specific vision for the project. Given your 30-day timeline and clear requirements, I want to ensure we align our approach to meet your goals effectively within your budget.

Do you have 10-15 minutes later this week for a brief discovery call? I’d like to understand your current website infrastructure and the volume of support queries you are looking to automate, so we can determine the best technical path forward.

Please let me know your availability, or feel free to suggest a time that works best for you.

Best regards,

[Your Name]
[Your Title]
[Your Company]
[Your Phone Number]


In [32]:
import sqlite3

conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

cursor.execute("""
UPDATE leads
SET follow_up_email = ?
WHERE email = ?
""", (sai_followup_email, lead["email"]))

conn.commit()

print("Sai's follow-up email saved successfully!")

conn.close()

Sai's follow-up email saved successfully!


In [33]:
conn = sqlite3.connect("lead_management.db")
cursor = conn.cursor()

cursor.execute("""
SELECT name, company, follow_up_email
FROM leads
WHERE email = ?
""", (lead["email"],))

result = cursor.fetchone()

print("Name:", result[0])
print("Company:", result[1])
print("Follow-up Email:")
print(result[2])

conn.close()

Name: sivanaga
Company: splenden
Follow-up Email:
Subject: Following up: AI-powered chatbot for Splenden

Hi Sivanaga,

Thank you for reaching out to discuss your interest in implementing an AI-powered chatbot for Splenden. 

Automating customer support queries is a great way to boost operational efficiency, and I would love to learn more about your specific vision for the project. Given your 30-day timeline and clear requirements, I want to ensure we align our approach to meet your goals effectively within your budget.

Do you have 10-15 minutes later this week for a brief discovery call? I’d like to understand your current website infrastructure and the volume of support queries you are looking to automate, so we can determine the best technical path forward.

Please let me know your availability, or feel free to suggest a time that works best for you.

Best regards,

[Your Name]
[Your Title]
[Your Company]
[Your Phone Number]
